# Generate All TrackNet Ball Candidates

This diagnostic notebook runs the same three-frame TrackNet detector and Hough-circle clustering as
`generate_ball_coords_json.ipynb`, but deliberately performs no cross-frame main-ball selection.

Every frame keeps all clustered candidates. The outputs are intended for candidate inspection only and
must not be passed directly to the trajectory/RANSAC notebooks as a single ball track.

In [ ]:
from __future__ import annotations

import json
import os
import sys
from collections import deque
from datetime import datetime, timezone
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np

os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

import torch
from tqdm.auto import tqdm

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name in {"scripts", "experiments"} and PROJECT_ROOT.parent.name == "cv":
    PROJECT_ROOT = PROJECT_ROOT.parents[1]
elif PROJECT_ROOT.name == "cv":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from cv.config import DEFAULT_BALL_MODEL_PATH, INPUT_DIR, OUTPUT_DIR as CV_OUTPUT_DIR
from cv.models.ball_tracking.general import postprocess
from cv.models.ball_tracking.model import BallTrackerNet

## 1. Configure Paths and Detection Parameters

In [ ]:
INPUT_VIDEO_PATH = INPUT_DIR / "real60.mp4"

CANDIDATE_BALL_MODEL_PATHS = [
    DEFAULT_BALL_MODEL_PATH,
    PROJECT_ROOT / "weights/model_best.pt",
]
BALL_MODEL_PATH = next(
    (path for path in CANDIDATE_BALL_MODEL_PATHS if path.exists()),
    CANDIDATE_BALL_MODEL_PATHS[0],
)

OUTPUT_DIR = CV_OUTPUT_DIR
CANDIDATE_JSON_PATH = OUTPUT_DIR / "ball_candidates_no_viterbi.json"
CANDIDATE_PNG_PATH = OUTPUT_DIR / "ball_candidates_no_viterbi.png"
CANDIDATE_VIDEO_PATH = OUTPUT_DIR / "ball_candidates_no_viterbi_overlay.mp4"

# Set a small value (for example 120) while iterating, or None for the full video.
MAX_FRAMES = None

# TrackNet predicts one of 256 grayscale classes per heatmap pixel.
BALLNESS_CLASS_THRESHOLD = 128
CONFIDENCE_PATCH_RADIUS = 3

# Hough may return overlapping circles around one heat blob. This distance is in 640x360 heatmap pixels.
CANDIDATE_CLUSTER_DISTANCE_PX = 10.0

print("video:", INPUT_VIDEO_PATH)
print("ball model:", BALL_MODEL_PATH)
print("candidate JSON:", CANDIDATE_JSON_PATH)
print("candidate PNG:", CANDIDATE_PNG_PATH)
print("candidate video:", CANDIDATE_VIDEO_PATH)

In [ ]:
if not INPUT_VIDEO_PATH.exists():
    raise FileNotFoundError(f"Input video not found: {INPUT_VIDEO_PATH}")
if not BALL_MODEL_PATH.exists():
    raise FileNotFoundError(
        f"TrackNet model not found: {BALL_MODEL_PATH}. Set BALL_MODEL_PATH before running."
    )

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 2. Load TrackNet

In [ ]:
def choose_torch_device():
    if torch.cuda.is_available():
        return "cuda"
    if torch.backends.mps.is_available():
        return "mps"
    return "cpu"


device = choose_torch_device()
model = BallTrackerNet()
state_dict = torch.load(BALL_MODEL_PATH, map_location="cpu", weights_only=True)
model.load_state_dict(state_dict)
model.to(device)
model.eval()
print("device:", device)

## 3. Extract and Cluster Per-frame Candidates

`raw_candidates` contains every Hough circle. `candidates` merges nearby circles into confidence-weighted
centers so that one heat blob appears as one diagnostic point. No information from adjacent frames is used.

In [ ]:
def extract_candidate_circles(feature_map, *, input_w=640, input_h=360, orig_w=1920, orig_h=1080):
    feature_map = feature_map.reshape((input_h, input_w))
    feature_map = (feature_map * 255).astype(np.uint8)
    _, heatmap = cv2.threshold(feature_map, 127, 255, cv2.THRESH_BINARY)

    circles = cv2.HoughCircles(
        heatmap,
        cv2.HOUGH_GRADIENT,
        dp=1,
        minDist=1,
        param1=50,
        param2=2,
        minRadius=2,
        maxRadius=7,
    )

    if circles is None or len(circles) == 0:
        return []

    scale_x = orig_w / input_w
    scale_y = orig_h / input_h
    raw_candidates = []
    for rank, circle in enumerate(circles[0], start=1):
        cx, cy, radius = circle
        raw_candidates.append({
            "raw_rank": rank,
            "pixel": [
                round((float(cx) + 0.5) * scale_x, 2),
                round((float(cy) + 0.5) * scale_y, 2),
            ],
            "heatmap_pixel": [round(float(cx), 2), round(float(cy), 2)],
            "radius": round(float(radius), 2),
        })
    return raw_candidates


def score_candidate(
    candidate,
    *,
    ballness,
    expected_heat,
    peak_probability,
    predicted_class,
    max_class_value,
    patch_radius=3,
):
    cx, cy = candidate["heatmap_pixel"]
    input_h, input_w = ballness.shape
    mx = int(np.clip(round(cx), 0, input_w - 1))
    my = int(np.clip(round(cy), 0, input_h - 1))
    y1, y2 = max(0, my - patch_radius), min(input_h, my + patch_radius + 1)
    x1, x2 = max(0, mx - patch_radius), min(input_w, mx + patch_radius + 1)

    ballness_patch = ballness[y1:y2, x1:x2]
    heat_patch = expected_heat[y1:y2, x1:x2]
    peak_patch = peak_probability[y1:y2, x1:x2]
    class_patch = predicted_class[y1:y2, x1:x2]

    candidate["confidence"] = round(float(ballness_patch.max().detach().cpu()), 4)
    candidate["heat_conf"] = round(float(heat_patch.max().detach().cpu()), 4)
    candidate["peak_class_probability"] = round(float(peak_patch.max().detach().cpu()), 4)
    candidate["predicted_intensity"] = round(
        float(class_patch.max().detach().cpu()) / max(max_class_value, 1),
        4,
    )
    candidate["score"] = candidate["confidence"]
    return candidate


def cluster_candidate_circles(raw_candidates, *, cluster_distance_px=CANDIDATE_CLUSTER_DISTANCE_PX):
    if not raw_candidates:
        return []

    remaining = sorted(raw_candidates, key=lambda item: item.get("score", 0.0), reverse=True)
    clusters = []
    while remaining:
        seed = remaining.pop(0)
        sx, sy = seed["heatmap_pixel"]
        members = [seed]
        keep = []

        for candidate in remaining:
            cx, cy = candidate["heatmap_pixel"]
            if float(np.hypot(cx - sx, cy - sy)) <= cluster_distance_px:
                members.append(candidate)
            else:
                keep.append(candidate)
        remaining = keep

        weights = np.array(
            [max(member.get("score", 0.0), 1e-6) for member in members],
            dtype=np.float64,
        )
        heatmap_points = np.array([member["heatmap_pixel"] for member in members], dtype=np.float64)
        pixel_points = np.array([member["pixel"] for member in members], dtype=np.float64)
        center_heatmap = np.average(heatmap_points, axis=0, weights=weights)
        center_pixel = np.average(pixel_points, axis=0, weights=weights)
        best_member = max(members, key=lambda item: item.get("score", 0.0))

        clusters.append({
            "rank": 0,
            "pixel": [round(float(center_pixel[0]), 2), round(float(center_pixel[1]), 2)],
            "heatmap_pixel": [
                round(float(center_heatmap[0]), 2),
                round(float(center_heatmap[1]), 2),
            ],
            "radius": round(float(np.mean([member.get("radius", 0.0) for member in members])), 2),
            "score": round(float(best_member.get("score", 0.0)), 4),
            "confidence": best_member.get("confidence"),
            "heat_conf": best_member.get("heat_conf"),
            "peak_class_probability": best_member.get("peak_class_probability"),
            "predicted_intensity": best_member.get("predicted_intensity"),
            "member_count": len(members),
            "raw_ranks": [member["raw_rank"] for member in members],
        })

    clusters.sort(key=lambda item: item.get("score", 0.0), reverse=True)
    for rank, cluster in enumerate(clusters, start=1):
        cluster["rank"] = rank
    return clusters

In [ ]:
def infer_candidates_3_frames(model, frames_3, *, device, orig_w, orig_h):
    oldest, middle, newest = frames_3
    input_w, input_h = 640, 360
    oldest = cv2.resize(oldest, (input_w, input_h))
    middle = cv2.resize(middle, (input_w, input_h))
    newest = cv2.resize(newest, (input_w, input_h))

    images = np.concatenate((newest, middle, oldest), axis=2).astype(np.float32) / 255.0
    images = np.rollaxis(images, 2, 0)
    batch = np.expand_dims(images, axis=0)

    with torch.no_grad():
        output_tensor = model(torch.from_numpy(batch).float().to(device))

    n_classes = int(output_tensor.shape[1])
    logits = output_tensor.reshape(1, n_classes, input_h, input_w)
    probabilities = torch.softmax(logits, dim=1)[0]
    class_values = torch.arange(
        n_classes,
        device=probabilities.device,
        dtype=probabilities.dtype,
    ).view(n_classes, 1, 1)
    expected_heat = (probabilities * class_values).sum(dim=0) / max(n_classes - 1, 1)
    ballness = probabilities[BALLNESS_CLASS_THRESHOLD:].sum(dim=0)
    peak_probability, predicted_class = probabilities.max(dim=0)
    feature_map = predicted_class.detach().cpu().numpy()

    raw_candidates = extract_candidate_circles(
        feature_map,
        input_w=input_w,
        input_h=input_h,
        orig_w=orig_w,
        orig_h=orig_h,
    )
    for candidate in raw_candidates:
        score_candidate(
            candidate,
            ballness=ballness,
            expected_heat=expected_heat,
            peak_probability=peak_probability,
            predicted_class=predicted_class,
            max_class_value=n_classes - 1,
            patch_radius=CONFIDENCE_PATCH_RADIUS,
        )

    candidates = cluster_candidate_circles(raw_candidates)
    post_x, post_y = postprocess(
        feature_map,
        input_w=input_w,
        input_h=input_h,
        orig_w=orig_w,
        orig_h=orig_h,
    )
    postprocess_pixel = None
    if post_x is not None and post_y is not None and 0 <= post_x <= orig_w and 0 <= post_y <= orig_h:
        postprocess_pixel = [round(float(post_x), 2), round(float(post_y), 2)]

    return {
        "raw_candidates": raw_candidates,
        "candidates": candidates,
        "postprocess_pixel": postprocess_pixel,
    }

## 4. Run Detection Without Cross-frame Selection

In [ ]:
capture = cv2.VideoCapture(str(INPUT_VIDEO_PATH))
if not capture.isOpened():
    raise RuntimeError(f"Could not open input video: {INPUT_VIDEO_PATH}")

fps = float(capture.get(cv2.CAP_PROP_FPS))
frame_width = int(capture.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT))
source_frame_count = int(capture.get(cv2.CAP_PROP_FRAME_COUNT))
target_frames = min(source_frame_count, MAX_FRAMES) if MAX_FRAMES is not None else source_frame_count

frame_buffer = deque(maxlen=3)
candidate_frames = []
progress = tqdm(total=target_frames, desc="TrackNet candidate detection")

for frame_index in range(target_frames):
    ok, frame = capture.read()
    if not ok:
        break

    frame_buffer.append(frame.copy())
    detection = {
        "raw_candidates": [],
        "candidates": [],
        "postprocess_pixel": None,
    }
    if len(frame_buffer) == 3:
        detection = infer_candidates_3_frames(
            model,
            list(frame_buffer),
            device=device,
            orig_w=frame_width,
            orig_h=frame_height,
        )

    candidate_frames.append({
        "frame_index": frame_index,
        "time_sec": round(frame_index / fps, 4) if fps > 0 else 0.0,
        "raw_candidates": detection["raw_candidates"],
        "candidates": detection["candidates"],
        "postprocess_pixel": detection["postprocess_pixel"],
    })
    progress.update(1)

capture.release()
progress.close()

raw_candidate_count = sum(len(item["raw_candidates"]) for item in candidate_frames)
clustered_candidate_count = sum(len(item["candidates"]) for item in candidate_frames)
frames_with_candidates = sum(bool(item["candidates"]) for item in candidate_frames)
frames_with_multiple_candidates = sum(len(item["candidates"]) > 1 for item in candidate_frames)
max_candidates_per_frame = max((len(item["candidates"]) for item in candidate_frames), default=0)

print(f"frames={len(candidate_frames):,}")
print(f"raw circles={raw_candidate_count:,}, clustered candidates={clustered_candidate_count:,}")
print(
    f"frames with candidates={frames_with_candidates:,}, "
    f"frames with multiple candidates={frames_with_multiple_candidates:,}, "
    f"max candidates/frame={max_candidates_per_frame}"
)

## 5. Write Candidate JSON

In [ ]:
candidate_counts = [len(item["candidates"]) for item in candidate_frames]
count_histogram = {
    str(count): candidate_counts.count(count)
    for count in sorted(set(candidate_counts))
}

payload = {
    "metadata": {
        "version": "ball-candidates-1.0.0",
        "source_video": INPUT_VIDEO_PATH.name,
        "source_video_path": str(INPUT_VIDEO_PATH),
        "ball_model_path": str(BALL_MODEL_PATH),
        "fps": fps,
        "video_width": frame_width,
        "video_height": frame_height,
        "source_total_frames": source_frame_count,
        "processed_frames": len(candidate_frames),
        "generated_at": datetime.now(timezone.utc).isoformat(),
        "algorithm": "tracknet_hough_clustered_candidates_without_cross_frame_selection",
        "confidence_method": f"softmax_probability_intensity_ge_{BALLNESS_CLASS_THRESHOLD}",
        "ballness_class_threshold": BALLNESS_CLASS_THRESHOLD,
        "confidence_patch_radius": CONFIDENCE_PATCH_RADIUS,
        "candidate_cluster_distance_heatmap_px": CANDIDATE_CLUSTER_DISTANCE_PX,
    },
    "summary": {
        "raw_candidate_count": raw_candidate_count,
        "clustered_candidate_count": clustered_candidate_count,
        "frames_with_candidates": frames_with_candidates,
        "frames_with_multiple_candidates": frames_with_multiple_candidates,
        "max_candidates_per_frame": max_candidates_per_frame,
        "mean_candidates_per_frame": round(
            clustered_candidate_count / max(len(candidate_frames), 1),
            4,
        ),
        "candidate_count_histogram": count_histogram,
    },
    "frames": candidate_frames,
}

CANDIDATE_JSON_PATH.write_text(
    json.dumps(payload, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print("wrote:", CANDIDATE_JSON_PATH, f"({CANDIDATE_JSON_PATH.stat().st_size / 1024:.1f} KB)")

## 6. Render the Multi-candidate Scatter Plot

Each candidate is an independent scatter point. Points from adjacent frames are intentionally not connected.
Color represents candidate confidence.

In [ ]:
scatter_frames = []
scatter_x = []
scatter_y = []
scatter_confidence = []
scatter_sizes = []

for frame_item in candidate_frames:
    for candidate in frame_item["candidates"]:
        pixel = candidate.get("pixel")
        if pixel is None:
            continue
        scatter_frames.append(frame_item["frame_index"])
        scatter_x.append(float(pixel[0]))
        scatter_y.append(float(pixel[1]))
        confidence = candidate.get("confidence")
        scatter_confidence.append(float(confidence) if confidence is not None else 0.0)
        scatter_sizes.append(10.0 + 2.0 * min(int(candidate.get("member_count") or 1), 8))

figure, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True, constrained_layout=True)
title_end = len(candidate_frames) - 1
figure.suptitle(f"Clustered TrackNet ball candidates: frames 0-{title_end}")

if scatter_frames:
    plot_options = {
        "c": scatter_confidence,
        "s": scatter_sizes,
        "cmap": "viridis",
        "vmin": 0.0,
        "vmax": 1.0,
        "alpha": 0.8,
        "linewidths": 0,
    }
    x_points = axes[0].scatter(scatter_frames, scatter_x, **plot_options)
    axes[1].scatter(scatter_frames, scatter_y, **plot_options)
    axes[2].scatter(
        scatter_frames,
        scatter_confidence,
        c=scatter_confidence,
        s=scatter_sizes,
        cmap="viridis",
        vmin=0.0,
        vmax=1.0,
        alpha=0.8,
        linewidths=0,
    )
    figure.colorbar(x_points, ax=axes[:2], label="candidate confidence", pad=0.01)
else:
    for axis in axes:
        axis.text(0.5, 0.5, "No candidates detected", ha="center", va="center", transform=axis.transAxes)

axes[0].set_ylabel("x pixel")
axes[1].set_ylabel("y pixel")
axes[2].set_ylabel("confidence")
axes[2].set_xlabel("frame")
axes[0].set_ylim(0, max(frame_width, 1))
axes[1].set_ylim(0, max(frame_height, 1))
axes[2].set_ylim(-0.02, 1.02)
for axis in axes:
    axis.grid(True, alpha=0.18)

figure.savefig(CANDIDATE_PNG_PATH, dpi=160, bbox_inches="tight")
plt.show()
print("wrote:", CANDIDATE_PNG_PATH)

## 7. Render All Clustered Candidates on Video

In [ ]:
def confidence_color(confidence):
    value = float(np.clip(confidence if confidence is not None else 0.0, 0.0, 1.0))
    return (int(255 * (1.0 - value)), int(80 + 175 * value), int(255 * value))


def render_candidate_overlay(
    input_video_path,
    output_video_path,
    frames,
    *,
    fps,
    frame_width,
    frame_height,
    marker_radius=7,
):
    capture = cv2.VideoCapture(str(input_video_path))
    if not capture.isOpened():
        raise RuntimeError(f"Could not open video for rendering: {input_video_path}")

    writer = cv2.VideoWriter(
        str(output_video_path),
        cv2.VideoWriter_fourcc(*"mp4v"),
        fps if fps > 0 else 25.0,
        (frame_width, frame_height),
    )
    if not writer.isOpened():
        capture.release()
        raise RuntimeError(f"Could not open video writer: {output_video_path}")

    total_frames = len(frames)
    progress = tqdm(total=total_frames, desc="Render clustered candidates")
    written = 0

    for frame_index, frame_item in enumerate(frames):
        ok, frame = capture.read()
        if not ok:
            break
        canvas = frame.copy()
        candidates = frame_item["candidates"]

        for candidate in candidates:
            pixel = candidate.get("pixel")
            if pixel is None:
                continue
            x, y = int(round(pixel[0])), int(round(pixel[1]))
            rank = int(candidate.get("rank") or 0)
            confidence = candidate.get("confidence")
            color = confidence_color(confidence)
            confidence_text = f"{float(confidence):.2f}" if confidence is not None else "n/a"
            label = f"#{rank} c={confidence_text}"

            cv2.circle(canvas, (x, y), marker_radius + 5, (0, 0, 0), -1, cv2.LINE_AA)
            cv2.circle(canvas, (x, y), marker_radius, color, -1, cv2.LINE_AA)
            cv2.circle(canvas, (x, y), marker_radius + 1, (255, 255, 255), 1, cv2.LINE_AA)
            cv2.putText(
                canvas,
                label,
                (x + 10, y - 8),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.48,
                (0, 0, 0),
                3,
                cv2.LINE_AA,
            )
            cv2.putText(
                canvas,
                label,
                (x + 10, y - 8),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.48,
                (255, 255, 255),
                1,
                cv2.LINE_AA,
            )

        hud = [
            f"frame {frame_index}/{total_frames - 1}",
            f"clustered candidates: {len(candidates)}",
            "all points are independent per-frame detections",
        ]
        for line_index, text in enumerate(hud):
            y_pos = 32 + line_index * 26
            cv2.putText(canvas, text, (20, y_pos), cv2.FONT_HERSHEY_SIMPLEX, 0.62, (0, 0, 0), 4, cv2.LINE_AA)
            cv2.putText(canvas, text, (20, y_pos), cv2.FONT_HERSHEY_SIMPLEX, 0.62, (255, 255, 255), 1, cv2.LINE_AA)

        writer.write(canvas)
        written += 1
        progress.update(1)

    capture.release()
    writer.release()
    progress.close()
    return written


rendered_frames = render_candidate_overlay(
    INPUT_VIDEO_PATH,
    CANDIDATE_VIDEO_PATH,
    candidate_frames,
    fps=fps,
    frame_width=frame_width,
    frame_height=frame_height,
)
print(f"wrote: {CANDIDATE_VIDEO_PATH} ({rendered_frames} frames)")

## 8. Interpretation

Multiple points at the same frame are expected. They may represent the real match ball, balls on other
courts, line-like heatmap artifacts, or duplicate visual structures. Use this notebook to inspect candidate
quality and tune detection/clustering. Use `generate_ball_coords_json.ipynb` when a single temporally
consistent trajectory is required.